# Combined Equation Model Exploration with Operator Composition

This notebook allows you to:
- Explore your DISCO model trained on combined physics equations (EULER, HEAT, DISP)
- Test operator composition methods (Greedy, Random, Exhaustive)
- Compare performance on out-of-distribution data
- Analyze operator usage patterns across physics types

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import h5py
from pathlib import Path
from tqdm import tqdm
import pandas as pd
from torch.utils.data import DataLoader
import random
from itertools import permutations
import math
import time

# Add project root to path
sys.path.append('/mnt/home/lserrano/disco-ball')
sys.path.append('/mnt/home/lserrano/disco-ball/tests/neural-operator-splitting')

from train.train_combined import DISCOLitModule# HDF5TemporalDataset
from src.utils.database import RelativeL2
from src.operators.disco import DISCOHouse
from operator_utils import sequential_operator_composition, strang_splitting_composition
from einops import rearrange

# Set up plotting
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
sns.set_style("whitegrid")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Configuration

In [ ]:
class HDF5TemporalDataset(torch.utils.data.Dataset):
    """Dataset for loading pre-computed trajectory data from HDF5 files"""
    
    def __init__(self, hdf5_files, input_frames=16, output_frames=16, 
                 sub_x=1, sub_t=1, split='train'):
        """
        Args:
            hdf5_files: List of HDF5 file paths to load data from
            input_frames: Number of input time frames
            output_frames: Number of output time frames  
            sub_x: Spatial subsampling factor
            sub_t: Temporal subsampling factor
            split: Dataset split ('train', 'val', 'test')
        """
        self.hdf5_files = hdf5_files if isinstance(hdf5_files, list) else [hdf5_files]
        self.input_frames = input_frames
        self.output_frames = output_frames
        self.sub_x = sub_x
        self.sub_t = sub_t
        self.split = split
        
        # Build file index for efficient access
        print("Building file index...")
        start_time = time.time()
        self.total_samples = self._build_file_index()
        index_time = time.time() - start_time
        print(f"Dataset length calculation took {index_time:.2f}s for {self.total_samples} samples")
        
        # Track loading times for performance assessment
        self.loading_times = []
        
    def _build_file_index(self):
        """Pre-compute file offsets for efficient __len__ and __getitem__"""
        self.file_offsets = []
        total_samples = 0
        
        for file_path in self.hdf5_files:
            if not os.path.exists(file_path):
                print(f"Warning: HDF5 file not found: {file_path}")
                continue
                
            try:
                with h5py.File(file_path, 'r') as f:
                    # Try different group names based on split
                    data_group = None
                    dataset_path = None
                    
                    # Check for split-specific groups first, then fall back to 'train'
                    possible_groups = [self.split, 'train', 'valid', 'test']
                    for group_name in possible_groups:
                        if group_name in f and 'pde_250-256' in f[group_name]:
                            data_group = group_name
                            dataset_path = f'{group_name}/pde_250-256'
                            break
                    
                    if data_group is None:
                        print(f"Warning: No valid dataset structure found in {file_path}. Checked groups: {possible_groups}")
                        continue
                        
                    n_samples = f[dataset_path].shape[0]
                    n_timesteps = f[dataset_path].shape[1]
                    
                    # Verify we have enough timesteps for input + output frames
                    min_timesteps_needed = (self.input_frames + self.output_frames) * self.sub_t
                    if n_timesteps < min_timesteps_needed:
                        print(f"Warning: Not enough timesteps in {file_path}. "
                              f"Need {min_timesteps_needed}, got {n_timesteps}")
                        continue
                    
                    self.file_offsets.append((file_path, total_samples, n_samples, dataset_path))
                    total_samples += n_samples
                    print(f"Added {n_samples} samples from {file_path} (using {dataset_path})")
                    
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
                continue
                
        if total_samples == 0:
            raise ValueError("No valid samples found in any HDF5 files!")
            
        return total_samples
    
    def _get_file_and_local_idx(self, idx):
        """Convert global index to file path and local index"""
        for file_path, offset, n_samples, dataset_path in self.file_offsets:
            if idx < offset + n_samples:
                local_idx = idx - offset
                return file_path, local_idx, dataset_path
        raise IndexError(f"Index {idx} out of range for dataset size {self.total_samples}")
    
    def __len__(self):
        return self.total_samples
    
    def __getitem__(self, idx):
        start_time = time.time()
        
        # Find which file and local index
        file_path, local_idx, dataset_path = self._get_file_and_local_idx(idx)
        
        try:
            with h5py.File(file_path, 'r') as f:
                # Get the group containing the data
                group_name = dataset_path.split('/')[0]
                
                # Load trajectory data - shape: (n_timesteps, n_spatial)
                trajectory = f[dataset_path][local_idx]
                
                # Load PDE parameters (alpha, beta, gamma) for this sample
                alpha = f[group_name]['alpha'][local_idx]
                beta = f[group_name]['beta'][local_idx]
                gamma = f[group_name]['gamma'][local_idx]
                
                # Sample temporal window randomly
                total_frames_needed = self.input_frames + self.output_frames
                max_start = (trajectory.shape[0] // self.sub_t) - total_frames_needed
                if max_start <= 0:
                    # If not enough frames, use what we have
                    start_idx = 0
                    available_frames = trajectory.shape[0] // self.sub_t
                    actual_input_frames = min(self.input_frames, available_frames // 2)
                    actual_output_frames = available_frames - actual_input_frames
                else:
                    start_idx = np.random.randint(0, max_start + 1)
                    actual_input_frames = self.input_frames
                    actual_output_frames = self.output_frames
                
                # Apply temporal subsampling and extract sequences
                #start_t = 0
                start_t = 0 #start_index
                input_end_t = start_t + actual_input_frames * self.sub_t
                output_end_t = input_end_t + actual_output_frames * self.sub_t
                
                input_seq = trajectory[start_t:input_end_t:self.sub_t, ::self.sub_x]
                output_seq = trajectory[input_end_t:output_end_t:self.sub_t, ::self.sub_x]
                
                # Add channel dimension and convert to torch tensors
                # Expected format: (time, channels, spatial)
                input_tensor = torch.from_numpy(input_seq).unsqueeze(-2).float()
                output_tensor = torch.from_numpy(output_seq).unsqueeze(-2).float()
                
                # Track loading time
                loading_time = time.time() - start_time
                if len(self.loading_times) < 1000:  # Collect first 1000 samples
                    self.loading_times.append(loading_time)
                
                return {
                    'input': input_tensor, 
                    'target': output_tensor,
                    'alpha': float(alpha),
                    'beta': float(beta),
                    'gamma': float(gamma)
                }
                
        except Exception as e:
            print(f"Error loading sample {idx} from {file_path}: {e}")
            # Return dummy data to avoid training crash
            dummy_input = torch.zeros(self.input_frames, 1, 256 // self.sub_x)
            dummy_output = torch.zeros(self.output_frames, 1, 256 // self.sub_x)
            return {'input': dummy_input, 'target': dummy_output}
    
    def get_loading_stats(self):
        """Return loading performance statistics"""
        if not self.loading_times:
            return {}
        
        return {
            'avg_loading_time': np.mean(self.loading_times),
            'min_loading_time': np.min(self.loading_times),
            'max_loading_time': np.max(self.loading_times),
            'samples_per_second': 1.0 / np.mean(self.loading_times),
            'total_samples_timed': len(self.loading_times)
        }

In [ ]:
# Model configuration - UPDATE THESE PATHS
#run_name = "DISCO_combined-physics-hdf5_solverrk4_adjFalse_h128_t3_steps1_initFalse_bs64_lr0.0005_hdf5_noise0.001_inframes16_outframes2_subx1_subt1"
#run_name = "DISCO_combined-physics-hdf5_solverrk4_adjFalse_h128_t3_steps1_initTrue_bs64_lr0.0005_hdf5_noise0_inframes16_outframes16_subx1_subt1"
#run_name = "DISCO_combined-physics-hdf5_solverrk4_adjFalse_h128_t3_steps1_initFalse_bs64_lr0.0005_hdf5_noise0_inframes16_outframes16_subx1_subt1"
#run_name="DISCO_combined-physics-hdf5_solverrk4_adjFalse_h128_t3_steps1_initTrue_bs64_lr0.0005_hdf5_noise0_inframes16_outframes16_subx1_subt1_resumed"



#run_name = "DISCO_combined-physics-hdf5_solverrk4_adjFalse_h128_t3_steps1_initTrue_bs64_lr0.0005_hdf5_noise0_inframes16_outframes16_subx1_subt1_20250902_140828"
#run_name = "DISCO_combined-physics-hdf5_solverrk4_adjFalse_h256_t3_steps1_initTrue_bs64_lr0.0005_hdf5_noise0_inframes16_outframes16_subx1_subt1"
run_name = "DISCO_combined-physics-hdf5_solverrk4_adjFalse_h128_t3_steps1_initTrue_bs64_lr0.0005_hdf5_noise0_inframes16_outframes16_subx1_subt1_20250904_115414"

MODEL_CHECKPOINT_PATH = f"/mnt/home/lserrano/disco-ball/outputs/{run_name}/last.ckpt"
# Data paths
DATA_DIR = "/mnt/home/lserrano/disco-ball/datasets/combined_equation/"
VALIDATION_FILES = {
    'EULER': f"{DATA_DIR}E_EULER_valid.h5",
    'HEAT': f"{DATA_DIR}E_HEAT_valid.h5",
    'DISP': f"{DATA_DIR}E_DISP_valid.h5"
}

TEST_FILES = {
    'EULER': f"{DATA_DIR}E_EULER_test.h5",
    'HEAT': f"{DATA_DIR}E_HEAT_test.h5",
    'DISP': f"{DATA_DIR}E_DISP_test.h5"
}

# Training files for operator encoding
TRAINING_FILES = {
    'EULER': f"{DATA_DIR}E_EULER_train_8192.h5",
    'HEAT': f"{DATA_DIR}E_HEAT_train_8192.h5",
    'DISP': f"{DATA_DIR}E_DISP_train_8192.h5"
}

# Test configuration
BATCH_SIZE = 16
N_INPUT_FRAMES = 16
N_OUTPUT_FRAMES = 100
SUB_X = 1
SUB_T = 1 #1

# Operator composition configuration
OPERATOR_CONFIG = {
    'num_operators': 64,  # Number of operators to encode
    'n_trajectories_per_operator': 1,  # Trajectories per operator (anti-forgetting)
    'max_operators': 5,  # Maximum operators in composition
    'min_improvement_threshold': 5.0,  # Minimum improvement % to add operator
    'n_input_frames': N_INPUT_FRAMES,
    'n_output_frames': N_OUTPUT_FRAMES
}

print("Configuration:")
print(f"  Model path: {MODEL_CHECKPOINT_PATH}")
print(f"  Data directory: {DATA_DIR}")
print(f"  Operator config: {OPERATOR_CONFIG}")

print("\nData files:")
for name, path in VALIDATION_FILES.items():
    exists = "✓" if os.path.exists(path) else "✗ (missing)"
    print(f"  Validation {name}: {exists}")

for name, path in TRAINING_FILES.items():
    exists = "✓" if os.path.exists(path) else "✗ (missing)"
    print(f"  Training {name}: {exists}")

## Load Model and Data

In [ ]:
def load_model_from_checkpoint(checkpoint_path):
    """Load DISCO model from Lightning checkpoint"""
    if not os.path.exists(checkpoint_path):
        print(f"Checkpoint not found: {checkpoint_path}")
        print("Please update MODEL_CHECKPOINT_PATH with your actual model path")
        return None, None
    
    try:
        lit_model = DISCOLitModule.load_from_checkpoint(checkpoint_path, map_location=device)
        lit_model.eval()
        
        model = lit_model.model.to(device)
        model.eval()
        
        print(f"Model loaded successfully from {checkpoint_path}")
        print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
        
        return model, lit_model
        
    except Exception as e:
        print(f"Error loading model: {e}")
        return None, None

# Load the model
model, lit_model = load_model_from_checkpoint(MODEL_CHECKPOINT_PATH)
relative_l2_error = RelativeL2()

if model is None:
    print("\nTo use this notebook, you need to:")
    print("1. Train a model using train_combined.py")
    print("2. Update MODEL_CHECKPOINT_PATH above with your checkpoint path")

In [ ]:
def create_dataset_for_equation(equation_type, split='val', files_dict=None):
    """Create dataset for a specific equation type"""
    files_dict = files_dict or VALIDATION_FILES
    
    if equation_type not in files_dict:
        print(f"Equation type {equation_type} not found in files dict")
        return None, None
        
    file_path = files_dict[equation_type]
    
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return None, None
    
    dataset = HDF5TemporalDataset(
        hdf5_files=[file_path],
        input_frames=N_INPUT_FRAMES,
        output_frames=N_OUTPUT_FRAMES,
        sub_x=SUB_X,
        sub_t=SUB_T,
        split=split
    )
    
    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )
    
    print(f"{equation_type} {split} dataset: {len(dataset)} samples")
    return dataloader, dataset

# Create validation datasets
val_dataloaders = {}
val_datasets = {}

for eq_type in VALIDATION_FILES.keys():
    loader, ds = create_dataset_for_equation(eq_type, 'val')
    if loader is not None:
        val_dataloaders[eq_type] = loader
        val_datasets[eq_type] = ds

print(f"\nLoaded {len(val_dataloaders)} validation datasets")


test_dataloaders = {}
test_datasets = {}

for eq_type in TEST_FILES.keys():
    loader, ds = create_dataset_for_equation(eq_type, 'test', files_dict=TEST_FILES)
    if loader is not None:
        test_dataloaders[eq_type] = loader
        test_datasets[eq_type] = ds


train_dataloaders = {}
train_datasets = {}

for eq_type in TRAINING_FILES.keys():
    loader, ds = create_dataset_for_equation(eq_type, 'train', files_dict=TRAINING_FILES)
    if loader is not None:
        train_dataloaders[eq_type] = loader
        train_datasets[eq_type] = ds

In [ ]:
eq_type="EULER"
num_integration_steps=1
N_OUTPUT_FRAMES=32

total_error = 0
test_size = 0
all_theta_latent = []
all_alpha = []
all_beta = []
all_gamma = []
all_errors = []

for batch in tqdm(val_dataloaders[eq_type]):
    inp, target = batch["input"], batch["target"]
    alpha, beta, gamma = batch['alpha'], batch['beta'], batch['gamma']
    all_alpha.append(alpha)
    all_beta.append(beta)
    all_gamma.append(gamma)
    
    inp = inp.squeeze(1)
    target = target.squeeze(1)
            
    inp = inp.to(device)
    target = target.to(device)
    state_labels = torch.tensor([0], device=inp.device)
        
    x_shape = inp.shape
    B, T, C = x_shape[:3]
    spatial = x_shape[3:]
    dim = len(spatial)
    
    n_sample = inp.shape[0]
    #pred, theta = autoregressive_predict(model, inp, n_pred=target.shape[1], device=device)
    
    #predict
    with torch.no_grad():
        # encode into 2 dimensional
        theta_latent, metadata= model.encode_theta_latent(inp, state_labels)

        ### WARNING
        #theta_latent = theta_latent[torch.randperm(B)]
        all_theta_latent.append(theta_latent.cpu())
        theta = model.decode_theta(theta_latent, dim)
        n_output_frames = N_OUTPUT_FRAMES
        #pred, metadata = model.solve_ode(inp[:, -1], theta, state_labels, dim, n_future_steps=n_output_frames,)# integration_time=4/256*n_output_frames, dt=4/256, predict_normed=False, metadata=metadata)
        pred, metadata = model.solve_ode(inp[:, -1], theta, state_labels, dim, n_future_steps=n_output_frames, integration_time=4/256, dt=4/256)#dt=4/256, predict_normed=False, metadata=metadata)

    rollout_error = relative_l2_error(pred, target[:, :n_output_frames]).item()
    
    # new
    sample_rollout_error = relative_l2_error(pred, target[:, :n_output_frames], ).item()
    x = rearrange(pred.clone(), "b ... -> b (...)")
    y = rearrange(target[:, :n_output_frames].clone(), "b ... -> b (...)")
    diff_norms = torch.linalg.norm(x - y, ord=2, dim=-1)
    y_norms = torch.linalg.norm(y, ord=2, dim=-1)
    sample_rollout_error = diff_norms / y_norms

    all_errors.append(sample_rollout_error)
    
    total_error+=rollout_error*n_sample
    test_size+=n_sample
    
all_errors = torch.cat(all_errors)
print('test error', total_error/test_size)

In [ ]:
# randperm 0.012
# 0.000838

In [ ]:
idx=8
for t in range(N_OUTPUT_FRAMES):
    plt.plot(pred[idx].squeeze().cpu().detach()[t])

In [ ]:
for t in range(N_OUTPUT_FRAMES):
    plt.plot(target[idx].squeeze().cpu().detach()[t])

In [ ]:
def plot_scatter(all_alpha, all_errors, run_name, title="train_t1"):
    plt.figure(figsize=(8, 6))
    plt.scatter(all_alpha, all_errors.cpu().detach(), alpha=0.6, s=30)
    plt.xlabel("Alpha"), plt.ylabel("Error"), plt.title(title), plt.grid(alpha=0.3)
    plt.yscale('log')  # Log scale for y-axis
    Path(f"plots/{run_name}").mkdir(parents=True, exist_ok=True)
    plt.savefig(f"plots/{run_name}/{title}.png", dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
plot_scatter(all_alpha, all_errors, run_name, title="train_t100")

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

def plot_3d_scatter(all_theta_latent, all_alpha, run_name):
    theta = all_theta_latent.cpu().detach() if hasattr(all_theta_latent, 'cpu') else all_theta_latent
    fig = plt.figure(figsize=(12, 4))
    
    # 3D plot
    ax1 = fig.add_subplot(131, projection='3d')
    scatter1 = ax1.scatter(theta[:, 0], theta[:, 1], theta[:, 2], c=all_alpha, cmap='viridis', s=20)
    ax1.set_xlabel('θ₀'), ax1.set_ylabel('θ₁'), ax1.set_zlabel('θ₂')
    
    # 2D projections
    ax2 = fig.add_subplot(132)
    scatter2 = ax2.scatter(theta[:, 0], theta[:, 1], c=all_alpha, cmap='viridis', s=20)
    ax2.set_xlabel('θ₀'), ax2.set_ylabel('θ₁')
    
    ax3 = fig.add_subplot(133)
    scatter3 = ax3.scatter(theta[:, 0], theta[:, 2], c=all_alpha, cmap='viridis', s=20)
    ax3.set_xlabel('θ₀'), ax3.set_ylabel('θ₂')
    
    plt.colorbar(scatter2, ax=[ax1, ax2, ax3], label='Alpha', shrink=0.8)
    
    plt.tight_layout()
    Path(f"plots/{run_name}").mkdir(parents=True, exist_ok=True)
    plt.savefig(f"plots/{run_name}/{run_name}.png", dpi=300, bbox_inches='tight')

# Usage: plot_3d_scatter(all_theta_latent, all_alpha, run_name)

In [ ]:
all_theta_latent = torch.cat(all_theta_latent)

In [ ]:
plot_3d_scatter(all_theta_latent, all_alpha, run_name="3d_train_t1")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D

# Your theta encodings from the neural network (replace with actual data)
theta = all_theta_latent# Shape (n, 3) - replace with your actual theta


In [ ]:
# 1. OOD Dataset Loader
def load_ood_dataset(ood_type, split='train'):
  """Load specific OOD dataset"""
  file_path = f"/mnt/home/lserrano/disco-ball/datasets/combined_equation/ood/{ood_type}_{split}_512.h5"
  dataset = HDF5TemporalDataset([file_path], N_INPUT_FRAMES, N_OUTPUT_FRAMES, SUB_X, SUB_T, split)
  dataloader = DataLoader(dataset, batch_size=8, shuffle=True)
  return dataloader

# 2. Simple Direct Prediction Test
def test_direct_prediction(model, dataloader, ood_name):
  """Test standard model prediction"""
  total_error = 0
  samples = 0
  prediction = []
  for batch in dataloader:
      inp, target = batch["input"].to(device), batch["target"].to(device)
      state_labels = torch.tensor([0], device=device)

      with torch.no_grad():
          pred, _ = model(inp, state_labels, n_future_steps=target.shape[1])#, integration_time=target.shape[1])
          prediction.append(pred.cpu())
          error = relative_l2_error(pred, target).item()
          total_error += error * inp.shape[0]
          samples += inp.shape[0]

  avg_error = total_error / samples
  print(f"{ood_name} Direct Prediction Error: {avg_error:.6f}")
  return avg_error, torch.cat(prediction)

# 3. Simple Theta Prediction Test  
def test_theta_prediction(model, dataloader, ood_name):
  """Test theta-based prediction"""
  total_error = 0
  samples = 0
  prediction = []
  for batch in dataloader:
      inp, target = batch["input"].to(device), batch["target"].to(device)
      state_labels = torch.tensor([0], device=device)

      with torch.no_grad():
          # Extract theta from encoder
          theta_latent, metadata = model.encode_theta_latent(inp, state_labels)
          theta = model.decode_theta(theta_latent, dim=1)

          # Predict using theta
          pred, _ = model.solve_ode(inp[:, -1], theta, state_labels, dim=1,
                                  n_future_steps=target.shape[1], )#integration_time=target.shape[1], dt=1,
                                   #predict_normed=False, metadata={})
          prediction.append(pred.cpu())
          error = relative_l2_error(pred, target).item()
          total_error += error * inp.shape[0]
          samples += inp.shape[0]

  avg_error = total_error / samples
  print(f"{ood_name} Theta Prediction Error: {avg_error:.6f}")
  return avg_error, torch.cat(prediction)

# 4. Simple Greedy Test (using first batch only for speed)
def test_greedy_composition(model, dataloader, encoded_operators, ood_name):
    """Test greedy search on first batch"""
    first_batch = next(iter(dataloader))
    
    inp, target = first_batch["input"].to(device), first_batch["target"].to(device)
    alpha = first_batch['alpha']
    beta = first_batch['beta']
    gamma = first_batch['gamma']
    theta_operators, _, metadata = encoded_operators
    pred_all = []
    for i in range(inp.shape[0]):
        print(f"EQUATION with alpha: {alpha[i]}, beta: {beta[i]}, gamma: {gamma[i]}")
        composition, _, pred = greedy_operator_selection(model, theta_operators, inp[i:i+1], target[i:i+1], max_operators=5)
        alphas = sum([metadata[op_id]["alpha"] for op_id in composition]).numpy()
        betas = sum([metadata[op_id]["beta"] for op_id in composition]).numpy()
        gammas = sum([metadata[op_id]["gamma"] for op_id in composition]).numpy()
        print(f"{ood_name} Best Greedy Composition: {composition}")
        print(f"Parameters found: alpha={alphas}, beta={betas}, gamma={gammas}")
        print(f"\n")
        pred_all.append(pred)
    return composition, torch.cat(pred_all)

# 5. Simple Random Test  
def test_random_composition(model, dataloader, encoded_operators, ood_name, num_compositions=1000, composition_lengths=[1,2,3,4,5]):
    """Test random search on first batch"""
    first_batch = next(iter(dataloader))
    inp, target = first_batch["input"].to(device), first_batch["target"].to(device)
    alpha = first_batch['alpha']
    beta = first_batch['beta']
    gamma = first_batch['gamma']
    theta_operators, _, metadata = encoded_operators
    pred_all = []
    for i in range(inp.shape[0]):
        print(f"EQUATION with alpha: {alpha[i]}, beta: {beta[i]}, gamma: {gamma[i]}")
        composition, _, pred = random_operator_selection(model, theta_operators, inp[i:i+1], target[i:i+1], num_compositions=num_compositions, composition_lengths=composition_lengths)
        alphas = sum([metadata[op_id]["alpha"] for op_id in composition]).numpy()
        betas = sum([metadata[op_id]["beta"] for op_id in composition]).numpy()
        gammas = sum([metadata[op_id]["gamma"] for op_id in composition]).numpy()
        print(f"{ood_name} Best Random Composition: {composition}")
        print(f"Parameters found: alpha={alphas}, beta={betas}, gamma={gammas}")
        print(f"\n")
        pred_all.append(pred)
    
    return composition, torch.cat(pred_all)
    
def test_nearest_param_selection(model, dataloader, encoded_operators, ood_name, num_integration_steps=5):
      """Test nearest parameter selection on first batch"""
      first_batch = next(iter(dataloader))

      inp, target = first_batch["input"].to(device), first_batch["target"].to(device)
      alpha = first_batch['alpha']
      beta = first_batch['beta']
      gamma = first_batch['gamma']
      theta_operators, _, metadata = encoded_operators

      # Ensure theta_operators is on the right device
      theta_operators = theta_operators.to(device)

      state_labels = torch.tensor([0], device=device)
      loss_fn = RelativeL2()

      prediction = []

      for i in range(inp.shape[0]):
          print(f"EQUATION with alpha: {alpha[i]}, beta: {beta[i]}, gamma: {gamma[i]}")

          # Find nearest parameter composition
          composition = nearest_param_selection(alpha[i], beta[i], gamma[i], metadata)

          if composition:
              # Test the composition
              model.eval()
              with torch.no_grad():
                  try:
                      pred = strang_splitting_composition(
                          inp[i:i+1, -1], state_labels, composition, theta_operators, model,
                          integration_time=4/256, n_future_steps=target.shape[1], num_integration_steps=num_integration_steps
                      )
                      if pred.ndim==3:
                          pred = pred.unsqueeze(0)
                      pred = rearrange(pred, 't b c h -> b t c h')
                      prediction.append(pred)
                      test_error = loss_fn(pred, target[i:i+1]).item()

                      # Get the actual parameters of the selected operators
                      found_alphas = [metadata[op_idx]["alpha"] for op_idx in composition]
                      found_betas = [metadata[op_idx]["beta"] for op_idx in composition]
                      found_gammas = [metadata[op_idx]["gamma"] for op_idx in composition]

                      # Convert to numpy if needed
                      found_alphas = [x.numpy() if hasattr(x, 'numpy') else x for x in found_alphas]
                      found_betas = [x.numpy() if hasattr(x, 'numpy') else x for x in found_betas]
                      found_gammas = [x.numpy() if hasattr(x, 'numpy') else x for x in found_gammas]

                      print(f"{ood_name} Nearest Param Composition: {composition}")
                      print(f"Parameters found: alpha={found_alphas}, beta={found_betas}, gamma={found_gammas}")
                      print(f"Test error: {test_error:.6f}")

                  except Exception as e:
                      print(f"Error applying composition {composition}: {e}")
          else:
              print(f"No suitable operator found (all target parameters are zero)")

          print(f"\n")

      return composition , torch.cat(prediction)

# 6. Test All OOD Datasets
def test_all_ood_datasets(ood_types = ['E_BG']):
  """Test all methods on all OOD datasets"""
  #ood_types = ['E_ALL', 'E_BG', 'E_ED', 'E_HE']
  #ood_types = ['E_HE']
  
  results = {}
  

  for ood_type in ood_types:
      print(f"\n=== Testing {ood_type} ===")

  
      dataloader = load_ood_dataset(ood_type)
      target = []
      for batch in dataloader:
          target.append(batch['target'])
          

      # Test direct prediction
      direct_error, pred_direct = test_direct_prediction(model, dataloader, ood_type)

      # Test theta prediction  
      #theta_error, pred_theta = test_theta_prediction(model, dataloader, ood_type)

      # Test compositions (if operators available)
      if encoded_operators:
          greedy_comp, pred_greedy = test_greedy_composition(model, dataloader, encoded_operators, ood_type)
          #random_comp, pred_random = test_random_composition(model, dataloader, encoded_operators, ood_type, composition_lengths=[2])
          #nearest_comp, pred_nearest = test_nearest_param_selection(model, dataloader, encoded_operators, ood_type, num_integration_steps=1)
      else:
          greedy_comp = random_comp = None

      results[ood_type] = {
          'direct_error': direct_error,
          #'theta_error': theta_error,
          #'greedy_composition': greedy_comp,
          #'random_composition': random_comp,
          #'pred_direct':pred_direct,
          #"pred_random":pred_random,
          #'pred_theta':pred_theta,
          'pred_greedy':pred_greedy,
          #'pred_nearest':pred_nearest,
          'target': torch.cat(target),
      }

      #except Exception as e:
      #    print(f"Error testing {ood_type}: {e}")
      #    results[ood_type] = {'error': str(e)}

  return results

In [ ]:
# Operator Encoding

def encode_operators_from_training_data(model, train_files, num_operators=20, 
                                       n_trajectories_per_operator=4):
    """Encode operators from training trajectories."""
    if model is None:
        print("Model not loaded")
        return None
    
    print(f"Encoding {num_operators} operators from training data...")
    
    # Create training datasets
    train_datasets = {}
    for eq_type, file_path in train_files.items():
        if os.path.exists(file_path):
            dataset = HDF5TemporalDataset(
                hdf5_files=[file_path],
                input_frames=N_INPUT_FRAMES,
                output_frames=N_OUTPUT_FRAMES,
                sub_x=1, sub_t=1, split='train'
            )
            train_datasets[eq_type] = dataset
            print(f"  {eq_type}: {len(dataset)} training samples")
    
    if not train_datasets:
        print("No training datasets available")
        return None
    
    # Collect trajectories for encoding
    all_trajectories = []
    operator_metadata = []
    
    trajectories_per_equation = num_operators // len(train_datasets)
    remaining = num_operators % len(train_datasets)

    total_collected=0
    for eq_idx, (eq_type, dataset) in enumerate(train_datasets.items()):
        ops_from_this_eq = trajectories_per_equation + (1 if eq_idx < remaining else 0)
        
        print(f"Encoding {ops_from_this_eq} operators from {eq_type}...")
        
        dataloader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=2)
        
        collected = 0
        target_trajectories = ops_from_this_eq * n_trajectories_per_operator
        
        for batch in dataloader:
            if collected >= target_trajectories:
                break
                
            input_seq = batch['input']
            alpha = batch['alpha']
            beta = batch['beta']
            gamma = batch['gamma']
            
            for sample_idx in range(input_seq.shape[0]):
                if collected >= target_trajectories:
                    break
                
                trajectory = input_seq[sample_idx:sample_idx+1]
                all_trajectories.append(trajectory)
                
                # Track operator metadata
                operator_idx = collected #// n_trajectories_per_operator
                #if collected % n_trajectories_per_operator == 0:
                operator_metadata.append({
                    'operator_id': total_collected,
                    'equation_type': eq_type,
                    'trajectory_indices': [],
                    'alpha':alpha[sample_idx:sample_idx+1],
                    'beta':beta[sample_idx:sample_idx+1],
                    'gamma':gamma[sample_idx:sample_idx+1]})   #   })
                
                operator_metadata[-1]['trajectory_indices'].append(len(all_trajectories) - 1)
                collected += 1
                total_collected +=1
    
    print(f"Collected {len(all_trajectories)} trajectories for {len(operator_metadata)} operators")
    
    # Encode all trajectories
    all_theta_latent = []
    all_theta = []
    
    state_labels = torch.tensor([0], device=device)
    encoding_batch_size = 32
    
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(all_trajectories), encoding_batch_size), desc="Encoding"):
            batch_trajectories = all_trajectories[i:i+encoding_batch_size]
            batch_input = torch.cat(batch_trajectories, dim=0).to(device)
            
            theta_latent_batch, _ = model.encode_theta_latent(batch_input, state_labels)
            theta_batch = model.decode_theta(theta_latent_batch, dim=1)
            
            all_theta_latent.append(theta_latent_batch.cpu())
            all_theta.append(theta_batch.cpu())
    
    all_theta_latent = torch.cat(all_theta_latent, dim=0)
    all_theta = torch.cat(all_theta, dim=0)

    return all_theta, all_theta_latent, operator_metadata
    
    ## Average parameters for each operator
    #num_unique_operators = len(operator_metadata)
    #theta_operators = torch.zeros(num_unique_operators, all_theta.shape[1])
    #theta_latent_operators = torch.zeros(num_unique_operators, all_theta_latent.shape[1])
    
    #for op_idx, op_meta in enumerate(operator_metadata):
    #    traj_indices = op_meta['trajectory_indices']
    #    theta_operators[op_idx] = all_theta[traj_indices].mean(dim=0)
    #    theta_latent_operators[op_idx] = all_theta_latent[traj_indices].mean(dim=0)
    
    #print(f"\nEncoded {num_unique_operators} operators:")
    #print(f"  Theta shape: {theta_operators.shape}")
    #print(f"  Theta latent shape: {theta_latent_operators.shape}")
    
    # Print distribution
    #eq_counts = {}
    #for op_meta in operator_metadata:
    #    eq_type = op_meta['equation_type']
    #    eq_counts[eq_type] = eq_counts.get(eq_type, 0) + 1
    
    #print(f"  Distribution: {eq_counts}")
    
    #return theta_operators, theta_latent_operators, operator_metadata

In [ ]:
def nearest_param_selection(target_alpha, target_beta, target_gamma, metadata):
  """
  For each non-zero target parameter, find the operator with the closest value.
  Combine all selected operators into a composition.
  
  Args:
      target_alpha: target alpha value (ignored if 0)
      target_beta: target beta value (ignored if 0)  
      target_gamma: target gamma value (ignored if 0)
      metadata: list/dict of operator metadata containing alpha, beta, gamma
      
  Returns:
      composition: list of operator indices (one for each non-zero target parameter)
  """
  import numpy as np

  # Convert targets to numpy if they're tensors
  if hasattr(target_alpha, 'numpy'):
      target_alpha = target_alpha.numpy()
  if hasattr(target_beta, 'numpy'):
      target_beta = target_beta.numpy()
  if hasattr(target_gamma, 'numpy'):
      target_gamma = target_gamma.numpy()

  composition = []

  # For each non-zero parameter, find the closest operator
  if target_alpha != 0:
      min_distance = float('inf')
      best_alpha_op = None

      for op_idx, op_metadata in enumerate(metadata):
          op_alpha = op_metadata['alpha']
          if hasattr(op_alpha, 'numpy'):
              op_alpha = op_alpha.numpy()

          distance = abs(target_alpha - op_alpha)
          if distance < min_distance:
              min_distance = distance
              best_alpha_op = op_idx

      if best_alpha_op is not None:
          composition.append(best_alpha_op)

  if target_beta != 0:
      min_distance = float('inf')
      best_beta_op = None

      for op_idx, op_metadata in enumerate(metadata):
          op_beta = op_metadata['beta']
          if hasattr(op_beta, 'numpy'):
              op_beta = op_beta.numpy()

          distance = abs(target_beta - op_beta)
          if distance < min_distance:
              min_distance = distance
              best_beta_op = op_idx

      if best_beta_op is not None:
          composition.append(best_beta_op)

  if target_gamma != 0:
      min_distance = float('inf')
      best_gamma_op = None

      for op_idx, op_metadata in enumerate(metadata):
          op_gamma = op_metadata['gamma']
          if hasattr(op_gamma, 'numpy'):
              op_gamma = op_gamma.numpy()

          distance = abs(target_gamma - op_gamma)
          if distance < min_distance:
              min_distance = distance
              best_gamma_op = op_idx

      if best_gamma_op is not None:
          composition.append(best_gamma_op)

  return composition

In [ ]:
def greedy_operator_selection(model, theta_operators, test_input, test_target, 
                             max_operators=5, min_improvement_threshold=5.0):
    """Greedy operator selection."""
    if model is None or theta_operators is None:
        return [], {}
    
    print(f"Running greedy operator selection...")
    print(f"Testing {theta_operators.shape[0]} operators, max length: {max_operators}")
    
    theta_operators = theta_operators.to(device)
    test_input = test_input.to(device)
    test_target = test_target.to(device)
    
    num_operators = theta_operators.shape[0]
    state_labels = torch.tensor([0], device=device)
    loss_fn = RelativeL2()
    
    # Random validation timestep
    #val_t = random.randint(0, test_input.shape[1] - 2)
    #x_val = test_input[:, val_t]
    #y_val = test_input[:, val_t + 1]
    x_val = rearrange(test_input[:, :-1], "b t c h -> (b t) c h")
    y_val = rearrange(test_input[:, 1:], "b t c h -> (b t) c h")
    
    current_composition = []
    current_best_error = float('inf')
    
    history = {
        'compositions': [],
        'errors': [],
        'method': 'greedy'
    }
    
    model.eval()
    
    for comp_length in range(1, max_operators + 1):
        print(f"\n--- Step {comp_length}: Testing all operators ---")
        
        best_error_for_step = float('inf')
        best_composition_for_step = None
        best_operator_added = None
        
        with torch.no_grad():
            for op_idx in range(num_operators):
                composition = current_composition + [op_idx]
                
                try:
                    #pred = sequential_operator_composition(
                    #    x_val, state_labels, composition, theta_operators, model,
                    #    integration_time=1.0, n_future_steps=1, num_integration_steps=1#5
                    #)
                     
                    pred = strang_splitting_composition(x_val, state_labels, composition, theta_operators, model,
                    integration_time=4/256, n_future_steps=1, num_integration_steps=5)#5)
                    
                    error = loss_fn(pred, y_val).item()
                    
                    history['compositions'].append(composition.copy())
                    history['errors'].append(error)
                    
                    if error < best_error_for_step:
                        best_error_for_step = error
                        best_composition_for_step = composition.copy()
                        best_operator_added = op_idx
                        
                except Exception as e:
                    continue
        
        if comp_length == 1:
            improvement = 0
            should_continue = True
        else:
            improvement = (current_best_error - best_error_for_step) / current_best_error * 100
            should_continue = improvement >= min_improvement_threshold
        
        print(f"  Best operator: {best_operator_added}, error: {best_error_for_step:.6f}")
        if comp_length > 1:
            print(f"  Improvement: {improvement:+.2f}%")
        
        if should_continue and best_error_for_step < current_best_error:
            current_composition = best_composition_for_step.copy()
            current_best_error = best_error_for_step
        else:
            print(f"  Stopping - insufficient improvement")
            break

    with torch.no_grad():
        pred = strang_splitting_composition(
                    test_input[:, -1], state_labels, current_composition, theta_operators, model,
                    integration_time=4/256, n_future_steps=N_OUTPUT_FRAMES, num_integration_steps=5
                )
        pred = rearrange(pred, 't b c h -> b t c h')
        test_error = loss_fn(pred, test_target).item()
                
    print(f"Greedy selection completed: {current_composition} leading to error: {test_error:.6f}")
    #print(f"\nGreedy selection completed: {current_composition} (error: {current_best_error:.6f})")
    return current_composition, history, pred

In [ ]:
def random_operator_selection(model, theta_operators, test_input, test_target, 
                             num_compositions=1000, composition_lengths=[1, 2, 3, 4, 5]):
    """Random search for operator composition."""
    if model is None or theta_operators is None:
        return [], {}
    
    print(f"Running random operator search...")
    print(f"Testing {num_compositions} random compositions")
    
    theta_operators = theta_operators.to(device)
    test_input = test_input.to(device)
    test_target = test_target.to(device)
    
    num_operators = theta_operators.shape[0]
    state_labels = torch.tensor([0], device=device)
    loss_fn = RelativeL2()
    
    val_t = random.randint(0, test_input.shape[1] - 2)
    #x_val = test_input[:, val_t]
    #y_val = test_input[:, val_t + 1]
    x_val = rearrange(test_input[:, :-1], "b t c h -> (b t) c h")
    y_val = rearrange(test_input[:, 1:], "b t c h -> (b t) c h")
    
    best_composition = []
    best_error = float('inf')
    
    history = {
        'compositions': [],
        'errors': [],
        'method': 'random',
        'total_tested': 0
    }
    
    model.eval()
    
    with torch.no_grad():
        for comp_idx in tqdm(range(num_compositions), desc="Random search"):
            comp_length = random.choice(composition_lengths)
            composition = random.sample(range(num_operators), comp_length)
            
            try:
                #pred = sequential_operator_composition(
                #    x_val, state_labels, composition, theta_operators, model,
                #    integration_time=1.0, n_future_steps=1, num_integration_steps=1#5
                #)
                
                pred = strang_splitting_composition(
                    x_val, state_labels, composition, theta_operators, model,
                    integration_time=4/256, n_future_steps=1, num_integration_steps=5 #5
                )
                
                error = loss_fn(pred, y_val).item()
                
                history['compositions'].append(composition.copy())
                history['errors'].append(error)
                history['total_tested'] += 1
                
                if error < best_error:
                    best_error = error
                    best_composition = composition.copy()
                    
            except Exception as e:
                continue

    with torch.no_grad():
        pred = strang_splitting_composition(
                    test_input[:, -1], state_labels, best_composition, theta_operators, model,
                    integration_time=4/256, n_future_steps=N_OUTPUT_FRAMES, num_integration_steps=5
                )
        pred = rearrange(pred, 't b c h -> b t c h')
        test_error = loss_fn(pred, test_target).item()
                
    print(f"Random search completed: {best_composition} leading to error: {test_error:.6f}")
    print(f"Successfully tested: {history['total_tested']}/{num_compositions}")
    
    return best_composition, history, pred

In [ ]:
#train_files = [TRAINING_FILES[key] for key in TRAINING_FILES.keys()]
theta_operators, theta_latent_operators, operator_metadata = encode_operators_from_training_data(model, TRAINING_FILES, num_operators=128*3, n_trajectories_per_operator=1)

In [ ]:
theta_latent_operators.shape

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D

# Your theta encodings from the neural network (replace with actual data)
theta = theta_latent_operators# Shape (n, 3) - replace with your actual theta

# Your metadata template with true parameters
template_info = operator_metadata

# Extract theta coordinates (neural network encodings)
theta_x = theta[:, 0]
theta_y = theta[:, 1] 
theta_z = theta[:, 2]

# Extract true parameters from metadata (handle PyTorch tensors)
true_alpha = [info['alpha'].item() if hasattr(info['alpha'], 'item') else info['alpha'] for info in template_info]
true_beta = [info['beta'].item() if hasattr(info['beta'], 'item') else info['beta'] for info in template_info]
true_gamma = [info['gamma'].item() if hasattr(info['gamma'], 'item') else info['gamma'] for info in template_info]

# Color mapping for equation types
color_map = {'EULER': '#FF6B6B', 'HEAT': '#4ECDC4', 'DISP': '#45B7D1', 'OTHER': '#96CEB4'}
colors = [color_map.get(info['equation_type'], color_map['OTHER']) for info in template_info]

# Create the 3D plot
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

# Plot points by equation type for proper legend
equation_types = list(set(info['equation_type'] for info in template_info))

for eq_type in equation_types:
    # Get indices for this equation type
    indices = [i for i, info in enumerate(template_info) if info['equation_type'] == eq_type]
    
    if indices:
        ax.scatter(theta_x[indices], theta_y[indices], theta_z[indices], 
                  c=color_map[eq_type], label=eq_type, s=60, alpha=0.7, 
                  edgecolors='black', linewidth=0.5)

# Only label a few representative points to avoid clutter
sample_indices = [0, len(template_info)//3, 2*len(template_info)//3, -1]  # Sample a few points
for i in sample_indices:
    if i < len(template_info):
        info = template_info[i]
        alpha_val = info['alpha'].item() if hasattr(info['alpha'], 'item') else info['alpha']
        beta_val = info['beta'].item() if hasattr(info['beta'], 'item') else info['beta']
        gamma_val = info['gamma'].item() if hasattr(info['gamma'], 'item') else info['gamma']
        
        # Only show the most relevant parameter (non-zero one)
        if alpha_val != 0:
            param_str = f"α={alpha_val:.3f}"
        elif beta_val != 0:
            param_str = f"β={beta_val:.3f}"
        elif gamma_val != 0:
            param_str = f"γ={gamma_val:.3f}"
        else:
            param_str = "zeros"
            
        label = f"{info['equation_type']}\n{param_str}"
        ax.text(theta_x[i], theta_y[i], theta_z[i], f"  {label}", 
                fontsize=9, ha='left', va='bottom',
                bbox=dict(boxstyle="round,pad=0.2", facecolor='white', alpha=0.8, edgecolor='gray'))

# Customize the plot
ax.set_xlabel('θ₁ (Neural Encoding Dim 1)', fontsize=12, labelpad=10)
ax.set_ylabel('θ₂ (Neural Encoding Dim 2)', fontsize=12, labelpad=10)
ax.set_zlabel('θ₃ (Neural Encoding Dim 3)', fontsize=12, labelpad=10)
ax.set_title('Neural Network Encodings (θ) vs True Parameters (α,β,γ)', fontsize=14, pad=20)

# Add legend
ax.legend(loc='upper left', bbox_to_anchor=(0.02, 0.98), fontsize=10)

# Improve visualization
ax.grid(True, alpha=0.3)
ax.xaxis.pane.fill = False
ax.yaxis.pane.fill = False
ax.zaxis.pane.fill = False

# Set viewing angle for better visualization
ax.view_init(elev=20, azim=45)

# Add text box with summary statistics
textstr = f"""Encoding Statistics:
θ₁: [{theta_x.min():.3f}, {theta_x.max():.3f}]
θ₂: [{theta_y.min():.3f}, {theta_y.max():.3f}]
θ₃: [{theta_z.min():.3f}, {theta_z.max():.3f}]

True Parameter Ranges:
α: [{min(true_alpha):.3f}, {max(true_alpha):.3f}]
β: [{min(true_beta):.3f}, {max(true_beta):.3f}]
γ: [{min(true_gamma):.3f}, {max(true_gamma):.3f}]"""

props = dict(boxstyle='round', facecolor='lightgray', alpha=0.8)
ax.text2D(0.02, 0.02, textstr, transform=ax.transAxes, fontsize=9,
          verticalalignment='bottom', bbox=props)

plt.tight_layout()
plt.show()

# Optional: Create a 2D correlation plot to see how encodings relate to true parameters
fig2, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot theta vs true alpha
for eq_type in equation_types:
    indices = [i for i, info in enumerate(template_info) if info['equation_type'] == eq_type]
    if indices:
        axes[0].scatter([true_alpha[i] for i in indices], [theta_x[i] for i in indices], 
                       c=color_map[eq_type], label=eq_type, s=80, alpha=0.7)

axes[0].set_xlabel('True α')
axes[0].set_ylabel('θ₁ (Encoding)')
axes[0].set_title('Neural Encoding vs True Alpha')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Plot theta vs true beta  
for eq_type in equation_types:
    indices = [i for i, info in enumerate(template_info) if info['equation_type'] == eq_type]
    if indices:
        axes[1].scatter([true_beta[i] for i in indices], [theta_y[i] for i in indices], 
                       c=color_map[eq_type], label=eq_type, s=80, alpha=0.7)

axes[1].set_xlabel('True β')
axes[1].set_ylabel('θ₂ (Encoding)')
axes[1].set_title('Neural Encoding vs True Beta')
axes[1].grid(True, alpha=0.3)

# Plot theta vs true gamma
for eq_type in equation_types:
    indices = [i for i, info in enumerate(template_info) if info['equation_type'] == eq_type]
    if indices:
        axes[2].scatter([true_gamma[i] for i in indices], [theta_z[i] for i in indices], 
                       c=color_map[eq_type], label=eq_type, s=80, alpha=0.7)

axes[2].set_xlabel('True γ')
axes[2].set_ylabel('θ₃ (Encoding)')
axes[2].set_title('Neural Encoding vs True Gamma')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Plots created!")
print(f"Neural encoding shape: {theta.shape}")
print("This shows how your neural network has learned to encode the true physical parameters.")

In [ ]:
encoded_operators = theta_operators, theta_latent_operators, operator_metadata 

In [ ]:
alphas = np.array([m['alpha'].numpy() for m in operator_metadata]).squeeze()
betas = np.array([m['beta'].numpy() for m in operator_metadata]).squeeze()
gammas = np.array([m['gamma'].numpy() for m in operator_metadata]).squeeze()
alphas.sort()
betas.sort()
gammas.sort()

In [ ]:
alphas.shape

In [ ]:
alphas[alphas>0]

In [ ]:
betas[betas>0]

In [ ]:
# Usage:
N_OUTPUT_FRAMES=50
results = test_all_ood_datasets(['E_BG'])

In [ ]:
# HEAT + BURGERS does not work it seems

In [ ]:
# try do do nearest neighbor parameter by parameter and see if it works

In [ ]:
#results['E_BG']

In [ ]:
target = results['E_BG']['target']

In [ ]:
#u_direct = results['E_BG']['pred_direct']

In [ ]:
u_composition = results['E_BG']['pred_nearest']

In [ ]:
idx=2

In [ ]:
for t in range(N_OUTPUT_FRAMES):
    plt.plot(target[idx].squeeze(1).cpu().detach()[t])

In [ ]:
for t in range(N_OUTPUT_FRAMES):
    plt.plot(u_composition[idx].squeeze(1).cpu().detach()[t])

In [ ]:
for t in range(N_OUTPUT_FRAMES):
    plt.plot(target[idx].squeeze(1).cpu().detach()[t])

## Operator Encoding

## Operator Selection Methods

In [ ]:
def compare_selection_methods(model, encoded_operators, test_sample):
    """Compare greedy vs random selection methods."""
    if model is None or encoded_operators is None:
        print("Model or operators not available")
        return {}
    
    theta_operators, theta_latent_operators, operator_metadata = encoded_operators
    
    print(f"\nComparing operator selection methods...")
    print(f"Using {theta_operators.shape[0]} operators")
    
    test_input, test_target = test_sample
    test_input = test_input.unsqueeze(0) if test_input.dim() == 3 else test_input[:1]
    test_target = test_target.unsqueeze(0) if test_target.dim() == 3 else test_target[:1]
    
    results = {}
    
    # 1. Direct prediction baseline
    print("\n--- Direct Prediction ---")
    with torch.no_grad():
        try:
            state_labels = torch.tensor([0], device=device)
            direct_pred, _ = model(
                test_input, state_labels, y=test_target,
                n_future_steps=test_target.shape[1]-1,
                integration_time=test_target.shape[1]-1
            )
            direct_error = RelativeL2()(direct_pred, test_target[:, 1:]).item()
            results['direct'] = {'error': direct_error, 'method': 'direct'}
            print(f"Direct prediction error: {direct_error:.6f}")
        except Exception as e:
            print(f"Direct prediction failed: {e}")
            results['direct'] = {'error': float('inf')}
    
    # 2. Greedy search
    print("\n--- Greedy Search ---")
    greedy_comp, greedy_hist = greedy_operator_selection(
        model, theta_operators, test_input, test_target,
        max_operators=min(5, theta_operators.shape[0])
    )
    
    if greedy_comp and greedy_hist['errors']:
        results['greedy'] = {
            'composition': greedy_comp,
            'error': min(greedy_hist['errors']),
            'evaluations': len(greedy_hist['errors']),
            'method': 'greedy'
        }
    
    # 3. Random search
    print("\n--- Random Search ---")
    random_comp, random_hist = random_operator_selection(
        model, theta_operators, test_input, test_target,
        num_compositions=min(200, theta_operators.shape[0] * 10)
    )
    
    if random_comp and random_hist['errors']:
        results['random'] = {
            'composition': random_comp,
            'error': min(random_hist['errors']),
            'evaluations': random_hist['total_tested'],
            'method': 'random'
        }
    
    # Print comparison
    print("\n" + "="*60)
    print("METHOD COMPARISON RESULTS")
    print("="*60)
    
    for method in ['direct', 'greedy', 'random']:
        if method in results and 'error' in results[method]:
            result = results[method]
            error = result['error']
            if error < float('inf'):
                comp_str = str(result.get('composition', 'N/A'))[:20]
                evals = result.get('evaluations', 'N/A')
                print(f"{method:10}: {error:.6f} | {comp_str:20} | {evals} evals")
            else:
                print(f"{method:10}: Failed")
    
    # Find best method
    valid_results = {k: v for k, v in results.items() 
                    if 'error' in v and v['error'] < float('inf')}
    
    if valid_results:
        best_method = min(valid_results.items(), key=lambda x: x[1]['error'])
        print(f"\nBest method: {best_method[0]} with error {best_method[1]['error']:.6f}")
        
        if 'direct' in valid_results:
            direct_error = valid_results['direct']['error']
            print("\nImprovement over direct prediction:")
            for method, result in valid_results.items():
                if method != 'direct' and 'composition' in result:
                    improvement = (direct_error - result['error']) / direct_error * 100
                    efficiency = improvement / result.get('evaluations', 1) * 100
                    print(f"  {method}: {improvement:+.2f}% ({efficiency:.4f}% per eval)")
    
    return results


## Test on Out-of-Distribution Data

In [ ]:
def test_operator_composition_on_data(model, encoded_operators, test_dataloader, 
                                     equation_name="Test", max_samples=3, 
                                     selection_method='greedy'):
    """Test operator composition on data."""
    if model is None or encoded_operators is None:
        return {}
    
    theta_operators, theta_latent_operators, operator_metadata = encoded_operators
    
    print(f"\nTesting {selection_method} composition on {equation_name} data...")
    print(f"Using {theta_operators.shape[0]} operators")
    
    loss_fn = RelativeL2()
    composition_results = []
    direct_results = []
    
    samples_tested = 0
    for batch_idx, batch in enumerate(test_dataloader):
        if samples_tested >= max_samples:
            break
            
        input_seq = batch['input']
        target_seq = batch['target']
        
        for sample_idx in range(min(input_seq.shape[0], max_samples - samples_tested)):
            print(f"\n--- Sample {samples_tested + 1} ---")
            
            test_input = input_seq[sample_idx:sample_idx+1].to(device)
            test_target = target_seq[sample_idx:sample_idx+1].to(device)
            
            # Direct prediction
            with torch.no_grad():
                try:
                    state_labels = torch.tensor([0], device=device)
                    direct_pred, _ = model(
                        test_input, state_labels, y=test_target,
                        n_future_steps=test_target.shape[1]-1,
                        integration_time=test_target.shape[1]-1
                    )
                    direct_error = loss_fn(direct_pred, test_target[:, 1:]).item()
                    
                    direct_results.append({
                        'sample_idx': samples_tested,
                        'error': direct_error
                    })
                    
                    print(f"  Direct error: {direct_error:.6f}")
                    
                except Exception as e:
                    print(f"  Direct prediction failed: {e}")
                    direct_error = float('inf')
            
            # Operator composition
            try:
                if selection_method == 'greedy':
                    best_composition, _ = greedy_operator_selection(
                        model, theta_operators, test_input, test_target,
                        max_operators=OPERATOR_CONFIG['max_operators']
                    )
                else:  # random
                    best_composition, _ = random_operator_selection(
                        model, theta_operators, test_input, test_target,
                        num_compositions=100
                    )
                
                if best_composition:
                    # Evaluate final composition on full sequence
                    with torch.no_grad():
                        x_test = test_input[:, -1]
                        pred_steps = []
                        current = x_test
                        
                        for step in range(test_target.shape[1]):
                            current = sequential_operator_composition(
                                current, state_labels, best_composition, 
                                theta_operators, model,
                                integration_time=1.0, n_future_steps=1,
                                num_integration_steps=1
                            )
                            pred_steps.append(current.unsqueeze(1))
                        
                        composition_pred = torch.cat(pred_steps, dim=1)
                        composition_error = loss_fn(composition_pred, test_target).item()
                        
                        improvement = (direct_error - composition_error) / direct_error * 100 if direct_error > 0 else 0
                        
                        composition_results.append({
                            'sample_idx': samples_tested,
                            'composition': best_composition,
                            'error': composition_error,
                            'improvement': improvement,
                            'method': selection_method
                        })
                        
                        print(f"  {selection_method.title()} composition: {best_composition}")
                        print(f"  Composition error: {composition_error:.6f}")
                        print(f"  Improvement: {improvement:+.2f}%")
                else:
                    print(f"  No valid composition found")
                    
            except Exception as e:
                print(f"  Composition failed: {e}")
            
            samples_tested += 1
    
    # Summary
    if composition_results and direct_results:
        comp_errors = [r['error'] for r in composition_results]
        direct_errors = [r['error'] for r in direct_results]
        improvements = [r['improvement'] for r in composition_results]
        
        print(f"\n{equation_name} Summary ({selection_method}):")
        print(f"  Samples: {samples_tested}")
        print(f"  Direct error: {np.mean(direct_errors):.6f} ± {np.std(direct_errors):.6f}")
        print(f"  Composition error: {np.mean(comp_errors):.6f} ± {np.std(comp_errors):.6f}")
        print(f"  Avg improvement: {np.mean(improvements):+.2f}%")
    
    return {
        'equation_name': equation_name,
        'method': selection_method,
        'samples_tested': samples_tested,
        'composition_results': composition_results,
        'direct_results': direct_results
    }

## Results Visualization

In [ ]:
def visualize_method_comparison(ood_test_results):
    """Visualize comparison between greedy and random methods."""
    if not ood_test_results:
        print("No results to visualize")
        return
    
    # Collect data
    greedy_data = []
    random_data = []
    
    for key, results in ood_test_results.items():
        if '_greedy' in key:
            greedy_data.extend(results['composition_results'])
        elif '_random' in key:
            random_data.extend(results['composition_results'])
    
    if not greedy_data or not random_data:
        print("Need both greedy and random results for comparison")
        return
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Error comparison
    greedy_errors = [r['error'] for r in greedy_data]
    random_errors = [r['error'] for r in random_data]
    
    axes[0].boxplot([greedy_errors, random_errors], labels=['Greedy', 'Random'])
    axes[0].set_ylabel('Composition Error')
    axes[0].set_title('Error Distribution by Method')
    axes[0].grid(alpha=0.3)
    
    # Improvement comparison
    greedy_improvements = [r['improvement'] for r in greedy_data]
    random_improvements = [r['improvement'] for r in random_data]
    
    axes[1].boxplot([greedy_improvements, random_improvements], labels=['Greedy', 'Random'])
    axes[1].set_ylabel('Improvement over Direct (%)')
    axes[1].set_title('Improvement Distribution')
    axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.7)
    axes[1].grid(alpha=0.3)
    
    # Scatter plot
    axes[2].scatter(greedy_errors, greedy_improvements, alpha=0.7, 
                   label='Greedy', s=50)
    axes[2].scatter(random_errors, random_improvements, alpha=0.7, 
                   label='Random', s=50)
    axes[2].set_xlabel('Composition Error')
    axes[2].set_ylabel('Improvement (%)')
    axes[2].set_title('Error vs Improvement')
    axes[2].axhline(y=0, color='red', linestyle='--', alpha=0.7)
    axes[2].legend()
    axes[2].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("\nMethod Comparison Summary:")
    print("=" * 40)
    print(f"{'Method':<10} {'Avg Error':<12} {'Avg Improvement':<15} {'Samples':<8}")
    print("-" * 40)
    
    for method_name, data in [('Greedy', greedy_data), ('Random', random_data)]:
        if data:
            avg_error = np.mean([r['error'] for r in data])
            avg_improvement = np.mean([r['improvement'] for r in data])
            n_samples = len(data)
            
            print(f"{method_name:<10} {avg_error:<12.6f} {avg_improvement:<15.2f} {n_samples:<8}")

if ood_test_results:
    visualize_method_comparison(ood_test_results)
else:
    print("No results available for visualization")

print("\n" + "="*70)
print("NOTEBOOK COMPLETE")
print("="*70)
print("\nWhat you can explore:")
print("1. Modify OPERATOR_CONFIG to test different numbers of operators")
print("2. Change selection methods (greedy vs random vs exhaustive)")
print("3. Test on truly out-of-distribution data with different parameter ranges")
print("4. Analyze which operators work best for different equation types")
print("5. Try the beam search from beam_search_gpu.py for more advanced selection")

In [ ]:

# Encode operators
encoded_operators = None
if model is not None:
    available_files = {k: v for k, v in TRAINING_FILES.items() if os.path.exists(v)}
    if available_files:
        print(f"Available training files: {list(available_files.keys())}")
        encoded_operators = encode_operators_from_training_data(
            model, available_files,
            num_operators=OPERATOR_CONFIG['num_operators'],
            n_trajectories_per_operator=OPERATOR_CONFIG['n_trajectories_per_operator']
        )
    else:
        print("No training files found - please update TRAINING_FILES paths")
else:
    print("Model not loaded - skipping operator encoding")

In [ ]:

# Run method comparison if everything is available
method_comparison = {}
if model is not None and encoded_operators is not None and val_dataloaders:
    print("\n" + "="*70)
    print("COMPARING OPERATOR SELECTION METHODS")
    print("="*70)
    
    # Get test sample
    sample_eq_type = list(val_dataloaders.keys())[0]
    sample_batch = next(iter(val_dataloaders[sample_eq_type]))
    sample_input = sample_batch['input'][0]
    sample_target = sample_batch['target'][0]
    
    print(f"Using {sample_eq_type} sample for comparison...")
    
    method_comparison = compare_selection_methods(
        model, encoded_operators, (sample_input, sample_target)
    )
else:
    print("Method comparison skipped - need model, operators, and validation data")

In [ ]:

# Test on validation data as OOD example
ood_test_results = {}

if model is not None and encoded_operators is not None and val_dataloaders:
    print("\n" + "="*70)
    print("TESTING OPERATOR COMPOSITION ON OUT-OF-DISTRIBUTION DATA")
    print("="*70)
    print("Using validation data as 'OOD' example...")
    
    for eq_type, dataloader in val_dataloaders.items():
        # Test both greedy and random
        for method in ['greedy', 'random']:
            results = test_operator_composition_on_data(
                model, encoded_operators, dataloader,
                equation_name=f"{eq_type}",
                max_samples=2,
                selection_method=method
            )
            ood_test_results[f"{eq_type}_{method}"] = results
else:
    print("OOD testing skipped - need model, operators, and validation data")